# CL Futures — Lab de validacion del motor de futuros

**Objetivo:** Validar end-to-end el motor de futuros con Crude Oil (CL/NYMEX).

- Sizing por contratos (floor de risk / stop_distance * point_value)
- Fees fijos por contrato ($2.50/lado, exchange + broker)
- P&L real: contracts * point_value * (exit - entry)
- pnl_pct como Return on Risk

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '..')

import pandas as pd
from strategies.examples.futures_ma_crossover import FuturesMACrossover
from core.backtest_runner import BacktestRunner
from utils.timeframe import Timeframe

## 1. Configuracion de la estrategia

CL specs:
- `tick_size`: $0.01 (minimo movimiento de precio)
- `tick_value`: $10.00 (valor de 1 tick)
- `point_value`: $1,000/punto (tick_value / tick_size = $10 / 0.01)
- `contract_size`: 1,000 barriles
- `fee`: $2.50/contrato/lado (exchange + broker)
- `slippage`: 2 ticks = $0.02 en precio = **$20/contrato** (porque $0.02 * $1,000/punto)

In [ ]:
strategy = FuturesMACrossover(
    symbol='CL',
    fast_period=10,
    slow_period=30,
    stop_points=2.0,    # $2 de stop -> $2,000 riesgo por contrato
    risk_pct=0.02,       # 2% del capital en riesgo por trade
    initial_capital=500_000,
    timeframe=Timeframe.D1
)

df = strategy.market_data
print(f'Datos: {len(df)} candles D1')
print(f'Rango: {df.index[0].date()} a {df.index[-1].date()}')
print(f'Precio: ${df["Close"].min():.2f} - ${df["Close"].max():.2f}')

## 2. Ejecutar backtest

In [ ]:
runner = BacktestRunner(strategy)
runner.run()
runner.print_summary()

## 3. Detalle de trades

Verificar que contracts, risk_usd y point_value estan correctos.

In [ ]:
results = runner.results
cols = ['entry_time', 'exit_time', 'avg_entry_price', 'exit_price',
        'contracts', 'risk_usd', 'gross_pnl', 'net_pnl', 'pnl_pct', 'capital_after']
results[cols].head(15)

In [ ]:
# Resumen rapido de contratos y riesgo
print(f'Total trades: {len(results)}')
print(f'Contracts promedio: {results["contracts"].mean():.1f}')
print(f'Risk USD promedio: ${results["risk_usd"].mean():,.0f}')
print(f'Fees totales: ${results["total_fees"].sum():,.2f}')
print(f'Slippage total: ${results["total_slippage"].sum():,.2f}')
print(f'Capital final: ${results["capital_after"].iloc[-1]:,.2f}')

## 4. Chart de trades — entradas y salidas

In [ ]:
runner.plot_trades(interactive=True)

## 5. Dashboards interactivos

In [ ]:
runner.plot_dashboards(interactive=True)